In [ ]:
%pip install nltk scikit-learn numpy pandas scipy matplotlib reportlab pypdf python-docx networkx plotly streamlit -q

In [ ]:
import os
import re
import time
import difflib
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import tkinter
from tkinter import filedialog
from collections import Counter
from xml.sax.saxutils import escape
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.chunk import ne_chunker
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import NMF
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import jensenshannon
from pypdf import PdfReader
from docx import Document as WordDocument
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.platypus import Image as PdfImage

nltk_packages = ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger",
                 "averaged_perceptron_tagger_eng", "maxent_ne_chunker", "maxent_ne_chunker_tab", "words", "vader_lexicon"]
for package in nltk_packages:
    nltk.download(package, quiet=True)

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_columns", 20)


def title(text):
    print()
    print("=" * 100)
    print(text.upper())
    print("=" * 100)


print("Setup is ready")

In [ ]:
DOCUMENT_FILES = []
OUTPUT_FOLDER = "output"

JOB_FILE = "auto"
RESUME_FILES = "auto"
VERSION_GROUPS = "auto"
SUMMARY_FILES = "auto"
REFERENCE_SUMMARY_FILE = "auto"

SEARCH_QUESTIONS = [
    "What is this document about?",
    "What skills are needed for the job?",
    "How many days, people or items are mentioned?",
    "Where does this take place?",
]

SUMMARY_SENTENCES = 4
SKILL_THRESHOLD = 0.60
VERSION_SIMILARITY = 0.35
TOPIC_GROUP_SIMILARITY = 0.12
DUPLICATE_SIMILARITY = 0.80
REQUIREMENT_SIMILARITY = 0.20

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print("Results will be saved in:", os.path.abspath(OUTPUT_FOLDER))
print("Put file paths in DOCUMENT_FILES, or leave it empty to choose files in a window")

In [ ]:
SUPPORTED_TYPES = (".txt", ".md", ".csv", ".pdf", ".docx")


def choose_files():
    window = tkinter.Tk()
    window.withdraw()
    window.attributes("-topmost", True)
    paths = filedialog.askopenfilenames(title="Choose the documents to analyse",
                                        filetypes=[("Documents", "*.txt *.md *.csv *.pdf *.docx"), ("All files", "*.*")])
    window.destroy()
    return list(paths)


def read_document(path):
    extension = os.path.splitext(path)[1].lower()
    if extension == ".pdf":
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if extension == ".docx":
        word_file = WordDocument(path)
        lines = [paragraph.text for paragraph in word_file.paragraphs]
        for table in word_file.tables:
            for row in table.rows:
                lines.append(" | ".join(box.text for box in row.cells))
        return "\n".join(lines)
    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


if not DOCUMENT_FILES:
    DOCUMENT_FILES = choose_files()

documents = {}
for path in sorted(DOCUMENT_FILES, key=lambda value: os.path.basename(value).lower()):
    file_name = os.path.basename(path)
    if not file_name.lower().endswith(SUPPORTED_TYPES):
        print("Skipped unsupported file:", file_name)
        continue
    text = read_document(path)
    if text.strip():
        documents[file_name] = text
    else:
        print("Skipped empty file (it may be a scanned PDF with no text):", file_name)

names = list(documents.keys())
if not names:
    raise ValueError("No documents chosen. Add file paths to DOCUMENT_FILES or pick .txt, .md, .csv, .pdf or .docx files in the window.")

loaded_table = pd.DataFrame([
    {"document": name, "type": os.path.splitext(name)[1], "characters": len(text)}
    for name, text in documents.items()
])
print(loaded_table.to_string(index=False))
print()
print("Total documents loaded:", len(names))

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
short_names = {"nlp": "natural language processing", "ml": "machine learning", "ai": "artificial intelligence",
               "js": "javascript", "hr": "human resources", "ui": "user interface"}
END_MARKS = (".", "!", "?", ":", ";")


def clean_text(text):
    text = text.replace("\t", " ").replace("\r", "")
    text = re.sub(r"[ ]+", " ", text)
    return text.strip()


def split_sentences(text):
    lines = [line.strip().strip("-*• ").strip() for line in clean_text(text).split("\n")]
    blocks = []
    buffer = ""
    for index, line in enumerate(lines):
        if not line:
            if buffer:
                blocks.append(buffer)
                buffer = ""
            continue
        buffer = (buffer + " " + line).strip()
        next_line = lines[index + 1] if index + 1 < len(lines) else ""
        keep_going = (not line.endswith(END_MARKS)) and next_line and (next_line[0].islower() or len(line) >= 75)
        if not keep_going:
            blocks.append(buffer)
            buffer = ""
    if buffer:
        blocks.append(buffer)
    sentences = []
    for block in blocks:
        for sentence in sent_tokenize(block):
            if len(sentence.strip()) > 2:
                sentences.append(sentence.strip())
    return sentences


def get_tokens(text):
    good_words = []
    for word in word_tokenize(text.lower()):
        if word.isalpha() and word not in stop_words:
            for part in short_names.get(word, word).split():
                good_words.append(lemmatizer.lemmatize(part))
    return good_words


def preprocess(text):
    return " ".join(get_tokens(text))


stemmer = PorterStemmer()


def preprocess_for_vectors(text):
    return " ".join(stemmer.stem(word) for word in get_tokens(text))


document_sentences = {name: split_sentences(text) for name, text in documents.items()}
sample = "The students were learning tokenization and stopword removal in the NLP classes."
print("Original :", sample)
print("Tokens   :", get_tokens(sample))
print()
for name in names:
    print(f"{name:28} sentences: {len(document_sentences[name])}")

In [ ]:
class TextVectorizer:
    def __init__(self):
        self.word_model = TfidfVectorizer(preprocessor=preprocess_for_vectors, ngram_range=(1, 2), sublinear_tf=True)
        self.char_model = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True, sublinear_tf=True)

    def fit(self, texts):
        self.word_model.fit(texts)
        self.char_model.fit(texts)
        return self

    def transform(self, texts):
        word_vectors = normalize(self.word_model.transform(texts).toarray())
        char_vectors = normalize(self.char_model.transform(texts).toarray())
        joined = np.hstack([word_vectors * np.sqrt(0.5), char_vectors * np.sqrt(0.5)])
        return normalize(joined)


def cosine_matrix(vectors_a, vectors_b):
    return np.clip(vectors_a @ vectors_b.T, 0, 1)


all_sentences = [sentence for name in names for sentence in document_sentences[name]]
main_vectorizer = TextVectorizer().fit(all_sentences + [documents[name] for name in names])
doc_vectors = main_vectorizer.transform([documents[name] for name in names])
doc_similarity_matrix = cosine_matrix(doc_vectors, doc_vectors)


def doc_similarity(first_name, second_name):
    return float(doc_similarity_matrix[names.index(first_name)][names.index(second_name)])


def text_similarity(first_text, second_text):
    vectors = main_vectorizer.transform([first_text, second_text])
    return float(cosine_matrix(vectors[:1], vectors[1:])[0][0])


print("Word features:", len(main_vectorizer.word_model.vocabulary_))
print("Character features:", len(main_vectorizer.char_model.vocabulary_))
print("Similarity of 'The dog slept on the rug' and 'A dog was sleeping on a rug':",
      round(text_similarity("The dog slept on the rug.", "A dog was sleeping on a rug."), 3))
print("Similarity of 'The dog slept on the rug' and 'Stock prices went up today':",
      round(text_similarity("The dog slept on the rug.", "Stock prices went up today."), 3))

In [ ]:
class VectorStore:
    def __init__(self, vectorizer):
        self.vectorizer = vectorizer
        self.texts = []
        self.sources = []
        self.vectors = None

    def add(self, texts, source):
        if not texts:
            return
        new_vectors = self.vectorizer.transform(texts)
        self.vectors = new_vectors if self.vectors is None else np.vstack([self.vectors, new_vectors])
        self.texts.extend(texts)
        self.sources.extend([source] * len(texts))

    def search(self, query, top_k=3, only_from=None, min_words=4):
        query_vector = self.vectorizer.transform([query])[0]
        scores = self.vectors @ query_vector
        results = []
        for position in np.argsort(scores)[::-1]:
            if only_from and self.sources[position] not in only_from:
                continue
            if len(self.texts[position].split()) < min_words:
                continue
            results.append({"source": self.sources[position], "text": self.texts[position], "score": round(float(scores[position]), 3)})
            if len(results) == top_k:
                break
        return results


store = VectorStore(main_vectorizer)
for name in names:
    store.add(document_sentences[name], name)
print("Sentences in the vector store:", len(store.texts))
print("Size of each vector:", store.vectors.shape[1])

In [ ]:
def name_key(file_name):
    stem = os.path.splitext(file_name)[0].lower()
    stem = re.sub(r"(v|ver|version|draft|rev)?\s*\d+", " ", stem)
    return re.sub(r"[_\-\s]+", " ", stem).strip()


def numbers_in_name(file_name):
    found = re.findall(r"\d+", file_name)
    return [int(number) for number in found] if found else [0]


def find_files(pattern):
    return [name for name in names if re.search(pattern, name.lower())]


if JOB_FILE == "auto":
    job_candidates = find_files(r"job|vacancy|opening|requirement|position")
    job_file = job_candidates[0] if job_candidates else None
else:
    job_file = JOB_FILE

resume_files = find_files(r"resume|\bcv\b|_cv|cv_|candidate") if RESUME_FILES == "auto" else list(RESUME_FILES)

if REFERENCE_SUMMARY_FILE == "auto":
    reference_candidates = find_files(r"reference|gold")
    reference_file = reference_candidates[0] if reference_candidates else None
else:
    reference_file = REFERENCE_SUMMARY_FILE

special_files = set(resume_files) | {job_file, reference_file}

if VERSION_GROUPS == "auto":
    name_groups = {}
    for name in names:
        if name not in special_files:
            name_groups.setdefault(name_key(name), []).append(name)
    version_groups = []
    for members in name_groups.values():
        if len(members) < 2:
            continue
        members.sort(key=numbers_in_name)
        neighbour_scores = [doc_similarity(a, b) for a, b in zip(members, members[1:])]
        if min(neighbour_scores) >= VERSION_SIMILARITY:
            version_groups.append(members)
else:
    version_groups = [list(group) for group in VERSION_GROUPS]

older_versions = {name for group in version_groups for name in group[:-1]}


def group_documents(file_list, min_similarity):
    if len(file_list) < 2:
        return [list(file_list)]
    vectors = doc_vectors[[names.index(name) for name in file_list]]
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=1 - min_similarity, metric="cosine", linkage="average")
    labels = model.fit_predict(vectors)
    groups = {}
    for name, label in zip(file_list, labels):
        groups.setdefault(label, []).append(name)
    return list(groups.values())


if SUMMARY_FILES == "auto":
    summary_candidates = [name for name in names if name not in special_files and name not in older_versions]
    if not summary_candidates:
        summary_candidates = names
    summary_groups = group_documents(summary_candidates, TOPIC_GROUP_SIMILARITY)
else:
    summary_groups = [list(SUMMARY_FILES)]

print("Job description file :", job_file)
print("Resume files         :", resume_files or "none found")
print("Reference summary    :", reference_file)
print("Version groups       :", version_groups or "none found")
print("Topic groups for summary:")
for group in summary_groups:
    print("   ", group)

In [ ]:
skill_taxonomy = {
    "JavaScript": ["javascript", "frontend scripting", "front end scripting", "vanilla javascript", "ecmascript", "typescript"],
    "React": ["react", "reactjs", "react js"],
    "Node.js": ["node", "nodejs", "node js", "express"],
    "Python": ["python", "python programming"],
    "Java": ["java", "spring boot"],
    "C and C++": ["c++", "cpp", "c programming"],
    "HTML": ["html", "html5"],
    "CSS": ["css", "css3", "tailwind", "bootstrap"],
    "SQL": ["sql", "mysql", "postgresql", "sql databases", "database queries", "oracle database"],
    "NoSQL": ["mongodb", "nosql", "redis"],
    "REST API": ["rest api", "rest apis", "api endpoints", "web services"],
    "Git": ["git", "github", "gitlab", "version control"],
    "Docker": ["docker", "docker containers", "kubernetes"],
    "Cloud Computing": ["aws", "azure", "cloud platforms", "cloud computing", "google cloud"],
    "Linux": ["linux", "unix", "shell scripting", "bash"],
    "Machine Learning": ["machine learning", "predictive modelling", "predictive modeling", "scikit learn", "classification models"],
    "Deep Learning": ["deep learning", "neural networks", "tensorflow", "pytorch"],
    "Natural Language Processing": ["natural language processing", "text mining", "text processing"],
    "Data Analysis": ["data analysis", "analysed data", "analyzed data", "pandas", "excel", "numpy"],
    "Data Visualisation": ["tableau", "power bi", "dashboards", "data visualisation", "data visualization", "matplotlib"],
    "Statistics": ["statistics", "statistical analysis", "probability"],
    "Testing": ["unit testing", "pytest", "selenium", "test automation"],
    "Project Management": ["project management", "agile", "scrum", "jira"],
    "Communication": ["communication", "communication skills", "presentation skills", "public speaking"],
    "Teamwork": ["teamwork", "team player", "work in a team", "collaboration"],
    "Leadership": ["leadership", "team lead", "mentoring"],
    "Problem Solving": ["problem solving", "analytical thinking", "critical thinking"],
    "UI Design": ["figma", "wireframes", "prototypes", "user interfaces", "ui design", "user interface design"],
    "Graphic Design": ["photoshop", "illustrator", "graphic design"],
    "User Research": ["user research", "user testing", "usability testing"],
    "Accounting": ["accounting", "bookkeeping", "tally", "financial reporting"],
    "Marketing": ["digital marketing", "seo", "social media marketing", "content marketing"],
    "Teaching": ["teaching", "lesson planning", "curriculum design"],
    "Polish Language": ["polish", "polish language"],
}
words_that_can_be_verbs = {"polish", "react", "node", "express", "teaching"}

phrase_to_skill = {}
for skill_name, phrases in skill_taxonomy.items():
    for phrase in phrases:
        phrase_to_skill[phrase] = skill_name
taxonomy_phrases = list(phrase_to_skill.keys())
skill_vectorizer = TextVectorizer().fit(taxonomy_phrases + all_sentences)
taxonomy_vectors = skill_vectorizer.transform(taxonomy_phrases)
noun_phrase_grammar = nltk.RegexpParser("PHRASE: {<JJ|NN.*>*<NN.*>}")


def tag_sentence(sentence):
    sentence = sentence.replace("-", " ").replace(".js", "js").replace("/", " , ")
    return nltk.pos_tag(word_tokenize(sentence))


def exact_skill_matches(tagged_words, found, rejected):
    lower_words = [word.lower() for word, tag in tagged_words]
    for size in [3, 2, 1]:
        for start in range(len(lower_words) - size + 1):
            phrase = " ".join(lower_words[start:start + size])
            if phrase not in phrase_to_skill:
                continue
            if size == 1 and phrase in words_that_can_be_verbs and tagged_words[start][1].startswith("VB"):
                rejected.append(f"'{phrase}' is used as a verb here, so it is not counted as a skill")
                continue
            if phrase_to_skill[phrase] not in found:
                found[phrase_to_skill[phrase]] = phrase


def similar_skill_matches(tagged_words, found):
    tree = noun_phrase_grammar.parse(tagged_words)
    for subtree in tree.subtrees(filter=lambda part: part.label() == "PHRASE"):
        phrase = " ".join(word for word, tag in subtree.leaves()).lower()
        if len(phrase) < 3:
            continue
        scores = cosine_matrix(skill_vectorizer.transform([phrase]), taxonomy_vectors)[0]
        best = int(np.argmax(scores))
        skill_name = phrase_to_skill[taxonomy_phrases[best]]
        if scores[best] >= SKILL_THRESHOLD and skill_name not in found:
            found[skill_name] = f"{phrase} (similarity {scores[best]:.2f})"


def extract_skills(text):
    found = {}
    rejected = []
    for sentence in split_sentences(text):
        tagged_words = tag_sentence(sentence)
        exact_skill_matches(tagged_words, found, rejected)
        similar_skill_matches(tagged_words, found)
    return found, rejected

print("Skill list ready with", len(skill_taxonomy), "skills and", len(taxonomy_phrases), "phrases")

In [ ]:
title("Operation 1: Document overview")
sentiment_tool = SentimentIntensityAnalyzer()


def count_syllables(word):
    word = word.lower()
    count = len(re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and count > 1:
        count -= 1
    return max(count, 1)


def readability(text):
    sentences = split_sentences(text)
    words = [word for word in word_tokenize(text) if word.isalpha()]
    if not sentences or not words:
        return 0.0
    syllables = sum(count_syllables(word) for word in words)
    return round(206.835 - 1.015 * len(words) / len(sentences) - 84.6 * syllables / len(words), 1)


def reading_level(score):
    if score >= 70:
        return "Easy"
    if score >= 50:
        return "Medium"
    if score >= 30:
        return "Hard"
    return "Very hard"


def sentiment(text):
    scores = [sentiment_tool.polarity_scores(sentence)["compound"] for sentence in split_sentences(text)]
    average = float(np.mean(scores)) if scores else 0.0
    label = "Positive" if average > 0.05 else "Negative" if average < -0.05 else "Neutral"
    return round(average, 3), label


keyword_model = TfidfVectorizer(preprocessor=preprocess, ngram_range=(1, 2), sublinear_tf=True)
keyword_matrix = keyword_model.fit_transform([documents[name] for name in names])
keyword_terms = np.array(keyword_model.get_feature_names_out())


def top_keywords(name, count=6):
    row = keyword_matrix[names.index(name)].toarray()[0]
    best = row.argsort()[::-1][:count]
    return [str(keyword_terms[i]) for i in best if row[i] > 0]


overview_rows = []
for name in names:
    text = documents[name]
    words = [word for word in word_tokenize(text) if word.isalpha()]
    score = readability(text)
    mood_score, mood = sentiment(text)
    overview_rows.append({
        "document": name,
        "words": len(words),
        "sentences": len(document_sentences[name]),
        "unique words": len(set(word.lower() for word in words)),
        "readability": score,
        "level": reading_level(score),
        "sentiment": mood_score,
        "mood": mood,
        "top keywords": ", ".join(top_keywords(name, 5)),
    })
overview_table = pd.DataFrame(overview_rows)
print(overview_table.to_string(index=False))

In [ ]:
title("Operation 2: Named entities and key facts")
month_pattern = r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?\s+\d{1,2}(?:,\s*\d{4})?\b|\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b(?:19|20)\d{2}\b"


entity_chunker = ne_chunker()


def is_real_entity(words, position, tagged, lowercase_words):
    if len(words) == 1 and position == 0:
        return False
    if all(word.lower() in lowercase_words for word in words):
        return False
    if any(word.lower() in phrase_to_skill for word in words) or " ".join(words).lower() in phrase_to_skill:
        return False
    next_position = position + len(words)
    if next_position < len(tagged) and tagged[next_position][0] == ":":
        return False
    return True


def find_entities(text, max_sentences=400):
    found = {"PERSON": Counter(), "ORGANIZATION": Counter(), "PLACE": Counter(), "DATE": Counter(),
             "NUMBER": Counter(), "MONEY": Counter(), "EMAIL": Counter(), "URL": Counter()}
    plain_text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+|https?://\S+", " ", text)
    lowercase_words = {word for word in word_tokenize(plain_text) if word.islower()}
    for sentence in split_sentences(text)[:max_sentences]:
        tagged = nltk.pos_tag(word_tokenize(sentence))
        tree = entity_chunker.parse(tagged)
        position = 0
        for part in tree:
            if not hasattr(part, "label"):
                position += 1
                continue
            words = [word for word, tag in part.leaves()]
            start = position
            position += len(words)
            if not is_real_entity(words, start, tagged, lowercase_words):
                continue
            entity = " ".join(words)
            if part.label() == "PERSON":
                found["PERSON"][entity] += 1
            elif part.label() in ("ORGANIZATION", "FACILITY"):
                found["ORGANIZATION"][entity] += 1
            elif part.label() in ("GPE", "LOCATION"):
                found["PLACE"][entity] += 1
    found["EMAIL"].update(re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", text))
    found["URL"].update(re.findall(r"https?://\S+|www\.\S+", text))
    found["DATE"].update(re.findall(month_pattern, text, flags=re.IGNORECASE))
    found["MONEY"].update(re.findall(r"(?:rs\.?|inr|usd|\$|₹|€|£)\s?\d[\d,]*(?:\.\d+)?", text, flags=re.IGNORECASE))
    found["NUMBER"].update(re.findall(r"\b\d+(?:\.\d+)?\s?(?:%|percent|days|years|months|hours|weeks)\b", text, flags=re.IGNORECASE))
    return found


document_entities = {name: find_entities(documents[name]) for name in names}
for name in names:
    print(name)
    for label, counter in document_entities[name].items():
        if counter:
            print(f"    {label:13}: {', '.join(item for item, count in counter.most_common(6))}")

In [ ]:
title("Operation 3: Similarity between all documents")
heatmap_path = os.path.join(OUTPUT_FOLDER, "similarity_heatmap.png")
figure_size = (max(7, len(names) * 0.75), max(6, len(names) * 0.65))
figure, axis = plt.subplots(figsize=figure_size)
picture = axis.imshow(doc_similarity_matrix, cmap="Blues", vmin=0, vmax=1)
axis.set_xticks(range(len(names)))
axis.set_yticks(range(len(names)))
axis.set_xticklabels(names, rotation=60, ha="right", fontsize=8)
axis.set_yticklabels(names, fontsize=8)
for row in range(len(names)):
    for col in range(len(names)):
        value = doc_similarity_matrix[row][col]
        axis.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=7, color="white" if value > 0.6 else "black")
axis.set_title("Document similarity (cosine of TF-IDF vectors)")
figure.colorbar(picture)
plt.tight_layout()
figure.savefig(heatmap_path, dpi=150)
plt.show()

pair_rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pair_rows.append({"document A": names[i], "document B": names[j], "similarity": round(float(doc_similarity_matrix[i][j]), 3)})
pair_table = pd.DataFrame(pair_rows, columns=["document A", "document B", "similarity"]).sort_values("similarity", ascending=False).reset_index(drop=True)
print("Most similar document pairs:")
print(pair_table.head(8).to_string(index=False))

In [ ]:
title("Operation 4: Topics found in the documents")
topic_count = max(1, min(5, len(names) // 2))
topic_model = NMF(n_components=topic_count, random_state=0, init="nndsvda", max_iter=600)
topic_weights = topic_model.fit_transform(keyword_matrix)
topic_words = []
for index, component in enumerate(topic_model.components_):
    words = [str(keyword_terms[i]) for i in component.argsort()[::-1][:6]]
    topic_words.append(words)
    print(f"Topic {index + 1}: {', '.join(words)}")
print()
topic_table = pd.DataFrame({
    "document": names,
    "main topic": [f"Topic {i + 1}" for i in topic_weights.argmax(axis=1)],
    "topic words": [", ".join(topic_words[i][:4]) for i in topic_weights.argmax(axis=1)],
})
print(topic_table.to_string(index=False))

In [ ]:
title("Operation 5: Semantic search over all documents")


def ask(question, top_k=3):
    print("Question:", question)
    results = store.search(question, top_k=top_k)
    for result in results:
        print(f"    {result['score']:.3f}  [{result['source']}]  {result['text']}")
    print()
    return results


search_results = {question: ask(question) for question in SEARCH_QUESTIONS}

In [ ]:
title("Operation 6: Shared or copied content between documents")


def same_version_group(first_name, second_name):
    return any(first_name in group and second_name in group for group in version_groups)


sentence_similarity = store.vectors @ store.vectors.T
close_pairs = np.argwhere(np.triu(sentence_similarity, 1) >= DUPLICATE_SIMILARITY)
shared_rows = []
for i, j in close_pairs:
    first_source = store.sources[i]
    second_source = store.sources[j]
    if first_source == second_source or same_version_group(first_source, second_source):
        continue
    if len(store.texts[i].split()) < 5:
        continue
    shared_rows.append({"similarity": round(float(sentence_similarity[i][j]), 3), "document A": first_source, "sentence A": store.texts[i],
                        "document B": second_source, "sentence B": store.texts[j]})
shared_table = pd.DataFrame(shared_rows)
if shared_table.empty:
    print("No copied or shared sentences found between different documents.")
else:
    print(shared_table.sort_values("similarity", ascending=False).to_string(index=False))

In [ ]:
title("Operation 7: Skills and competences in every document")
document_skills = {name: extract_skills(documents[name])[0] for name in names}
skills_table = pd.DataFrame([
    {"document": name, "skills found": len(skills), "skills": ", ".join(skills.keys()) or "none"}
    for name, skills in document_skills.items()
])
print(skills_table.to_string(index=False))

In [ ]:
MATCH_LEVEL = 0.60
CHANGE_LEVEL = 0.40
GAP_PENALTY = -0.6
NUMBERING = r"^\s*((unit|section|chapter|part|article|clause|step|week|module|lesson)\s*)?(\d+(\.\d+)*|[ivx]+)\s*[:.)\-]\s*"


def strip_numbering(sentence):
    return re.sub(NUMBERING, "", sentence, flags=re.IGNORECASE)


def pair_score(similarity):
    if similarity >= MATCH_LEVEL:
        return 2.0
    if similarity >= CHANGE_LEVEL:
        return 1.0
    return -1.0


def numbers_in(text):
    return set(re.findall(r"\b\d+(?:\.\d+)?%?", text))


def content_difference(old_sentence, new_sentence):
    old_words = set(get_tokens(old_sentence))
    new_words = set(get_tokens(new_sentence))

    def really_new(word, other_words):
        return not any(difflib.SequenceMatcher(None, word, other).ratio() >= 0.8 for other in other_words)

    added = sorted(word for word in new_words - old_words if really_new(word, old_words))
    removed = sorted(word for word in old_words - new_words if really_new(word, new_words))
    numbers_changed = numbers_in(old_sentence) != numbers_in(new_sentence)
    return added, removed, numbers_changed


def needleman_wunsch(old_sentences, new_sentences, vectorizer):
    if not old_sentences or not new_sentences:
        steps = [("REMOVED", s, "", 0.0) for s in old_sentences] + [("ADDED", "", s, 0.0) for s in new_sentences]
        return steps, np.zeros((1, 1))
    similarity = cosine_matrix(vectorizer.transform(old_sentences), vectorizer.transform(new_sentences))
    rows = len(old_sentences) + 1
    cols = len(new_sentences) + 1
    score = np.zeros((rows, cols))
    for i in range(1, rows):
        score[i][0] = i * GAP_PENALTY
    for j in range(1, cols):
        score[0][j] = j * GAP_PENALTY
    for i in range(1, rows):
        for j in range(1, cols):
            diagonal = score[i - 1][j - 1] + pair_score(similarity[i - 1][j - 1])
            up = score[i - 1][j] + GAP_PENALTY
            left = score[i][j - 1] + GAP_PENALTY
            score[i][j] = max(diagonal, up, left)
    steps = []
    i = rows - 1
    j = cols - 1
    while i > 0 or j > 0:
        if i > 0 and j > 0 and score[i][j] == score[i - 1][j - 1] + pair_score(similarity[i - 1][j - 1]):
            value = float(similarity[i - 1][j - 1])
            added, removed, numbers_changed = content_difference(old_sentences[i - 1], new_sentences[j - 1])
            if value >= MATCH_LEVEL and not added and not removed and not numbers_changed:
                steps.append(("SAME", old_sentences[i - 1], new_sentences[j - 1], value))
            elif value >= CHANGE_LEVEL:
                steps.append(("CHANGED", old_sentences[i - 1], new_sentences[j - 1], value))
            else:
                steps.append(("ADDED", "", new_sentences[j - 1], value))
                steps.append(("REMOVED", old_sentences[i - 1], "", value))
            i -= 1
            j -= 1
        elif i > 0 and score[i][j] == score[i - 1][j] + GAP_PENALTY:
            steps.append(("REMOVED", old_sentences[i - 1], "", 0.0))
            i -= 1
        else:
            steps.append(("ADDED", "", new_sentences[j - 1], 0.0))
            j -= 1
    steps.reverse()
    return steps, score


def semantic_relation(steps):
    labels = {step[0] for step in steps}
    if not labels & {"SAME", "CHANGED"}:
        return "No Relation (the documents share no information)"
    old_has_more = "REMOVED" in labels
    new_has_more = "ADDED" in labels
    for label, old_sentence, new_sentence, value in steps:
        if label == "CHANGED":
            added, removed, numbers_changed = content_difference(old_sentence, new_sentence)
            if numbers_changed or (added and removed):
                old_has_more = True
                new_has_more = True
            elif added:
                new_has_more = True
            elif removed:
                old_has_more = True
    if not old_has_more and not new_has_more:
        return "Semantic Equivalence (same information)"
    if not old_has_more:
        return "Semantic Inclusion (old version is fully inside the new version)"
    if not new_has_more:
        return "Semantic Inclusion (new version is fully inside the old version)"
    return "Semantic Overlap (common core with unique parts on both sides)"


def word_changes(old_sentence, new_sentence):
    old_words = old_sentence.split()
    new_words = new_sentence.split()
    removed = []
    added = []
    for tag, a_start, a_end, b_start, b_end in difflib.SequenceMatcher(None, old_words, new_words).get_opcodes():
        if tag in ("replace", "delete"):
            removed.extend(old_words[a_start:a_end])
        if tag in ("replace", "insert"):
            added.extend(new_words[b_start:b_end])
    return " ".join(removed), " ".join(added)


def lexical_diff_count(old_lines, new_lines):
    return sum(1 for line in difflib.ndiff(old_lines, new_lines) if line.startswith(("+ ", "- ")))


def compare_documents(old_name, new_name, show=True):
    old_text = documents[old_name]
    new_text = documents[new_name]
    old_sentences = [strip_numbering(s) for s in split_sentences(old_text)]
    new_sentences = [strip_numbering(s) for s in split_sentences(new_text)]
    steps, matrix = needleman_wunsch(old_sentences, new_sentences, main_vectorizer)
    counts = Counter(step[0] for step in steps)
    old_keywords = set(top_keywords(old_name, 15))
    new_keywords = set(top_keywords(new_name, 15))
    old_skills = set(document_skills.get(old_name, {}))
    new_skills = set(document_skills.get(new_name, {}))
    result = {
        "old": old_name,
        "new": new_name,
        "similarity": round(doc_similarity(old_name, new_name), 3),
        "relation": semantic_relation(steps),
        "steps": steps,
        "counts": {label: counts.get(label, 0) for label in ["SAME", "CHANGED", "ADDED", "REMOVED"]},
        "lexical lines flagged": lexical_diff_count(split_sentences(old_text), split_sentences(new_text)),
        "keywords only in old": sorted(old_keywords - new_keywords),
        "keywords only in new": sorted(new_keywords - old_keywords),
        "shared keywords": sorted(old_keywords & new_keywords),
        "skills removed": sorted(old_skills - new_skills),
        "skills added": sorted(new_skills - old_skills),
        "numbers removed": sorted(numbers_in(old_text) - numbers_in(new_text)),
        "numbers added": sorted(numbers_in(new_text) - numbers_in(old_text)),
    }
    if show:
        print_comparison(result)
    return result


def print_comparison(result):
    print("-" * 100)
    print(f"{result['old']}  ->  {result['new']}")
    print("Overall similarity :", result["similarity"])
    print("Relation           :", result["relation"])
    print("Semantic changes   :", result["counts"])
    real_changes = result["counts"]["CHANGED"] + result["counts"]["ADDED"] + result["counts"]["REMOVED"]
    print(f"A normal line diff flags {result['lexical lines flagged']} lines, the semantic diff finds {real_changes} real changes")
    print("Keywords only in old:", ", ".join(result["keywords only in old"]) or "none")
    print("Keywords only in new:", ", ".join(result["keywords only in new"]) or "none")
    if result["skills added"] or result["skills removed"]:
        print("Skills added:", ", ".join(result["skills added"]) or "none", "| Skills removed:", ", ".join(result["skills removed"]) or "none")
    if result["numbers added"] or result["numbers removed"]:
        print("Numbers added:", ", ".join(result["numbers added"]) or "none", "| Numbers removed:", ", ".join(result["numbers removed"]) or "none")
    print()
    for label, old_sentence, new_sentence, value in result["steps"]:
        if label == "SAME":
            print(f"[SAME    {value:.2f}] {new_sentence}")
        elif label == "CHANGED":
            removed, added = word_changes(old_sentence, new_sentence)
            print(f"[CHANGED {value:.2f}] {old_sentence}")
            print(f"{'':15}-> {new_sentence}")
            print(f"{'':15}   words removed: '{removed or 'none'}'  words added: '{added or 'none'}'")
        elif label == "ADDED":
            print(f"[ADDED       ] {new_sentence}")
        else:
            print(f"[REMOVED     ] {old_sentence}")
    print()


print("Comparison tools are ready. Use compare_documents('old_file', 'new_file') on any two documents.")

In [ ]:
title("Operation 8: Semantic version comparison (Needleman-Wunsch)")
version_results = []
evolution_rows = []
if not version_groups:
    print("No version groups found. Name files like report_v1.pdf and report_v2.pdf, or set VERSION_GROUPS in the settings cell.")
for group in version_groups:
    print("Version group:", " -> ".join(group))
    for old_name, new_name in zip(group, group[1:]):
        result = compare_documents(old_name, new_name)
        version_results.append(result)
        evolution_rows.append({"from": old_name, "to": new_name, "similarity": result["similarity"], **result["counts"],
                               "line diff noise": result["lexical lines flagged"], "relation": result["relation"].split(" (")[0]})
if evolution_rows:
    evolution_table = pd.DataFrame(evolution_rows)
    print("How the documents changed over time:")
    print(evolution_table.to_string(index=False))
else:
    evolution_table = pd.DataFrame()

In [ ]:
title("Operation 9: Compare any two documents you choose")
FIRST_FILE = version_groups[0][0] if version_groups else names[0]
SECOND_FILE = version_groups[0][-1] if version_groups else names[-1]
custom_comparison = compare_documents(FIRST_FILE, SECOND_FILE)

In [ ]:
title("Operation 10: OCR and spelling error robustness")
clean_sentence = "Text preprocessing including tokenization, lemmatization, stemming and stopword removal."
noisy_sentence = "Text preprocesing includng tokenizaton, lemmatizaton, stemming and stopword remval."
other_sentence = "Every employee gets 15 days of paid annual leave each year."
noisy_steps, noisy_score = needleman_wunsch([clean_sentence], [noisy_sentence], main_vectorizer)
print("Clean vs same text with OCR errors :", round(text_similarity(clean_sentence, noisy_sentence), 3), "-> counted as", noisy_steps[0][0])
print("Clean vs a different topic         :", round(text_similarity(clean_sentence, other_sentence), 3))

In [ ]:
title("Operation 11: Resume and job matching")
gender_words = r"\b(he|she|him|her|his|hers|mr|mrs|ms|miss|male|female|man|woman)\b"


def remove_personal_details(text):
    lines = [line for line in text.split("\n") if line.strip()]
    if lines and len(lines[0].split()) <= 4 and not re.search(r"\d|:", lines[0]):
        lines = lines[1:]
    text = "\n".join(lines)
    text = re.sub(r"(?im)^\s*(name|gender|sex|age|date of birth|dob|nationality|religion|caste|marital status|address)\s*:.*$", "", text)
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "", text)
    text = re.sub(r"\+?\d[\d\s-]{8,}\d", "", text)
    return re.sub(gender_words, "", text, flags=re.IGNORECASE)


def requirement_coverage(job_text, resume_text):
    requirements = [s for s in split_sentences(job_text) if len(s.split()) >= 7]
    resume_sentences = split_sentences(resume_text)
    if not requirements or not resume_sentences:
        return 0.0, []
    similarity = cosine_matrix(main_vectorizer.transform(requirements), main_vectorizer.transform(resume_sentences))
    rows = []
    for index, requirement in enumerate(requirements):
        best = int(similarity[index].argmax())
        rows.append({"requirement": requirement, "best resume line": resume_sentences[best], "score": round(float(similarity[index][best]), 3)})
    covered = sum(1 for row in rows if row["score"] >= REQUIREMENT_SIMILARITY)
    return covered / len(rows), rows


def match_resume(resume_text, job_text):
    job_skills = extract_skills(job_text)[0]
    safe_text = remove_personal_details(resume_text)
    resume_skills, rejected = extract_skills(safe_text)
    needed = set(job_skills)
    have = set(resume_skills)
    skill_score = len(needed & have) / len(needed) if needed else 0.0
    coverage, coverage_rows = requirement_coverage(job_text, safe_text)
    text_score = text_similarity(job_text, safe_text)
    final_score = 0.6 * skill_score + 0.25 * coverage + 0.15 * text_score if needed else 0.6 * coverage + 0.4 * text_score
    return {
        "skill_score": round(skill_score, 3),
        "coverage": round(coverage, 3),
        "text_score": round(text_score, 3),
        "final_score": round(final_score, 3),
        "matched": sorted(needed & have),
        "missing": sorted(needed - have),
        "extra": sorted(have - needed),
        "evidence": resume_skills,
        "rejected": rejected,
        "coverage_rows": coverage_rows,
    }


match_results = {}
ranking = pd.DataFrame()
if job_file and resume_files:
    job_skills = extract_skills(documents[job_file])[0]
    print("Job file:", job_file)
    print("Skills the job needs:", ", ".join(job_skills) or "none from the skill list, so matching uses requirement coverage")
    for name in resume_files:
        match_results[name] = match_resume(documents[name], documents[job_file])
    ranking = pd.DataFrame([
        {"rank": 0, "resume": name, "final score": r["final_score"], "skill score": r["skill_score"],
         "requirement coverage": r["coverage"], "text score": r["text_score"], "matched skills": len(r["matched"])}
        for name, r in match_results.items()
    ]).sort_values("final score", ascending=False).reset_index(drop=True)
    ranking["rank"] = range(1, len(ranking) + 1)
    print()
    print(ranking.to_string(index=False))
    for name in ranking["resume"]:
        result = match_results[name]
        print()
        print("Resume:", name)
        print("    Matched skills :", ", ".join(result["matched"]) or "none")
        print("    Missing skills :", ", ".join(result["missing"]) or "none")
        print("    Extra skills   :", ", ".join(result["extra"]) or "none")
        for skill, evidence in result["evidence"].items():
            print(f"        {skill:26} found as: {evidence}")
        for reason in result["rejected"]:
            print("    Rejected:", reason)
        weakest = sorted(result["coverage_rows"], key=lambda row: row["score"])[:2]
        for row in weakest:
            print(f"    Weak requirement ({row['score']}): {row['requirement']}")
else:
    print("Resume matching skipped. Add a file with 'job' in its name and files with 'resume' or 'cv' in their names.")

In [ ]:
title("Operation 12: Fairness check with counterfactual resumes")
swap_pairs = [("he", "she"), ("him", "her"), ("his", "her"), ("mr", "ms"), ("male", "female"), ("man", "woman")]
replacement_names = ["Alex Morgan", "Priya Sharma", "Mohammed Ali", "Chen Wei", "Maria Garcia"]


def make_counterfactual(text, new_name):
    lines = text.split("\n")
    first_line = next((line for line in lines if line.strip()), "").strip()
    people = []
    if first_line and len(first_line.split()) <= 4 and not re.search(r"\d|:", first_line):
        people.append(first_line)
    name_line = re.search(r"(?im)^\s*name\s*:\s*(.+)$", text)
    if name_line:
        people.append(name_line.group(1).strip())
    for person in set(people):
        if person:
            text = text.replace(person, new_name)
            first_word = person.split()[0].lower()
            text = re.sub(rf"\b{re.escape(first_word)}\b", new_name.split()[0].lower(), text)
    for first, second in swap_pairs:
        text = re.sub(rf"\b{first}\b", second, text, flags=re.IGNORECASE)
    return text


fairness_rows = []
if match_results:
    for index, name in enumerate(resume_files):
        original_score = match_results[name]["final_score"]
        new_name = replacement_names[index % len(replacement_names)]
        swapped_score = match_resume(make_counterfactual(documents[name], new_name), documents[job_file])["final_score"]
        difference = abs(original_score - swapped_score)
        fairness_rows.append({"resume": name, "original score": original_score, "name used in test": new_name,
                              "score after swap": swapped_score, "difference": round(difference, 4), "fair": bool(difference < 0.01)})
fairness_table = pd.DataFrame(fairness_rows)
if fairness_table.empty:
    print("Fairness check skipped because no resumes were matched.")
else:
    print(fairness_table.to_string(index=False))
    print()
    print("Fairness check passed" if fairness_table["fair"].all() else "Fairness check failed, please review the matching")

In [ ]:
def textrank_scores(similarity, damping=0.85, rounds=100):
    matrix = similarity.copy()
    np.fill_diagonal(matrix, 0)
    row_sums = matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    matrix = matrix / row_sums
    size = len(matrix)
    scores = np.ones(size) / size
    for _ in range(rounds):
        new_scores = (1 - damping) / size + damping * matrix.T @ scores
        if np.abs(new_scores - scores).sum() < 1e-6:
            break
        scores = new_scores
    return scores


def extract_evidence(file_list, max_sentences=4, redundancy_limit=0.5):
    sentences = []
    sources = []
    for name in file_list:
        for sentence in document_sentences[name]:
            tags = [tag for word, tag in nltk.pos_tag(word_tokenize(sentence))]
            has_verb = any(tag.startswith("VB") or tag == "MD" for tag in tags)
            if len(sentence.split()) >= 5 and has_verb:
                sentences.append(sentence)
                sources.append(name)
    if not sentences:
        return []
    vectors = main_vectorizer.transform(sentences)
    similarity = cosine_matrix(vectors, vectors)
    rank = textrank_scores(similarity)
    centre = normalize(vectors.mean(axis=0, keepdims=True))
    centrality = (vectors @ centre.T).ravel()
    position_bonus = np.array([1.0 if i == 0 or sources[i] != sources[i - 1] else 0.0 for i in range(len(sentences))])
    scores = 0.6 * rank / rank.max() + 0.3 * centrality / max(centrality.max(), 1e-9) + 0.1 * position_bonus
    chosen = []
    for position in np.argsort(scores)[::-1]:
        if any(similarity[position][other] > redundancy_limit for other in chosen):
            continue
        chosen.append(position)
        if len(chosen) == max_sentences:
            break
    chosen.sort()
    return [{"sentence": sentences[p], "source": sources[p], "score": round(float(scores[p]), 3)} for p in chosen]


short_forms = {
    "in order to": "to", "due to the fact that": "because", "a large number of": "many", "at this point in time": "now",
    "in the event that": "if", "with regard to": "about", "is able to": "can", "are able to": "can",
    "a majority of": "most", "prior to": "before", "is expected to": "will likely", "are expected to": "will likely",
    "is becoming": "is", "at the present time": "now", "for the purpose of": "for",
}
starting_words = ["however", "moreover", "in addition", "also", "furthermore", "therefore", "so", "thus", "finally", "overall"]


def rewrite_sentence(sentence):
    text = re.sub(r"\([^)]*\)", "", sentence)
    for long_form, short_form in short_forms.items():
        text = re.sub(rf"\b{long_form}\b", short_form, text, flags=re.IGNORECASE)
    for word in starting_words:
        if text.lower().startswith(word + ","):
            text = text[len(word) + 1:].strip()
            break
    text = strip_numbering(re.sub(r"\s+", " ", text).strip())
    if text and not text.endswith((".", "!", "?")):
        text = text.rstrip(":;,") + "."
    return text[0].upper() + text[1:] if text else text


def join_sentences(sentences):
    joined = []
    skip_next = False
    for index, sentence in enumerate(sentences):
        if skip_next:
            skip_next = False
            continue
        if index + 1 < len(sentences):
            following = sentences[index + 1]
            shared = set(get_tokens(sentence)) & set(get_tokens(following))
            if len(shared) >= 2 and len(sentence.split()) + len(following.split()) < 35:
                joined.append(sentence.rstrip(".") + ", and " + following[0].lower() + following[1:].rstrip(".") + ".")
                skip_next = True
                continue
        joined.append(sentence)
    return joined


def summarize(file_list, max_sentences=SUMMARY_SENTENCES):
    evidence = extract_evidence(file_list, max_sentences=max_sentences)
    rewritten = [rewrite_sentence(item["sentence"]) for item in evidence]
    summary = " ".join(join_sentences(rewritten))
    trace = [(rewritten[i], evidence[i]["source"]) for i in range(len(evidence))]
    return summary, trace


title("Operation 13: Short summary of every document")
single_summaries = {name: summarize([name], max_sentences=2)[0] for name in names}
for name, summary in single_summaries.items():
    print(f"{name}:")
    print("   ", summary or "(too short to summarise)")

In [ ]:
title("Operation 14: Multi-document summary for each topic group")
group_summaries = []
for group in summary_groups:
    summary, trace = summarize(group, max_sentences=SUMMARY_SENTENCES if len(group) > 1 else 3)
    group_summaries.append({"files": group, "summary": summary, "trace": trace})
    print("Documents:", ", ".join(group))
    print("Summary  :", summary)
    print("Where each part came from:")
    for sentence, source in trace:
        print(f"    [{source}] {sentence}")
    print()

In [ ]:
title("Operation 15: Summary quality (ROUGE and vector score)")


def ngrams(words, size):
    return Counter(tuple(words[i:i + size]) for i in range(len(words) - size + 1))


def f_score(overlap, candidate_total, reference_total):
    precision = overlap / max(candidate_total, 1)
    recall = overlap / max(reference_total, 1)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return round(precision, 3), round(recall, 3), round(f1, 3)


def rouge_n(candidate, reference, size):
    candidate_grams = ngrams(get_tokens(candidate), size)
    reference_grams = ngrams(get_tokens(reference), size)
    overlap = sum((candidate_grams & reference_grams).values())
    return f_score(overlap, sum(candidate_grams.values()), sum(reference_grams.values()))


def longest_common_subsequence(words_a, words_b):
    table = [[0] * (len(words_b) + 1) for _ in range(len(words_a) + 1)]
    for i in range(1, len(words_a) + 1):
        for j in range(1, len(words_b) + 1):
            if words_a[i - 1] == words_b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[-1][-1]


def rouge_l(candidate, reference):
    candidate_words = get_tokens(candidate)
    reference_words = get_tokens(reference)
    return f_score(longest_common_subsequence(candidate_words, reference_words), len(candidate_words), len(reference_words))


def vector_score(candidate, reference):
    candidate_vectors = main_vectorizer.transform(split_sentences(candidate))
    reference_vectors = main_vectorizer.transform(split_sentences(reference))
    similarity = cosine_matrix(candidate_vectors, reference_vectors)
    precision = float(similarity.max(axis=1).mean())
    recall = float(similarity.max(axis=0).mean())
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return round(precision, 3), round(recall, 3), round(f1, 3)


def evaluate_summary(candidate, reference):
    return pd.DataFrame(
        [rouge_n(candidate, reference, 1), rouge_n(candidate, reference, 2), rouge_l(candidate, reference), vector_score(candidate, reference)],
        index=["ROUGE-1", "ROUGE-2", "ROUGE-L", "Vector similarity"], columns=["precision", "recall", "f1"])


evaluation = pd.DataFrame()
evaluated_group = None
if reference_file and group_summaries:
    reference_text = documents[reference_file]
    best_group = max(group_summaries, key=lambda item: text_similarity(item["summary"], reference_text) if item["summary"] else 0)
    evaluated_group = best_group["files"]
    evaluation = evaluate_summary(best_group["summary"], reference_text)
    print("Reference file:", reference_file)
    print("Compared with the summary of:", ", ".join(evaluated_group))
    print(evaluation)
else:
    print("No reference summary found, so ROUGE was skipped. Add a file with 'reference' in its name to turn it on.")
    print("Self check: each single document summary compared with its full document")
    self_rows = []
    for name in names:
        if single_summaries[name]:
            self_rows.append({"document": name, "ROUGE-1 recall": rouge_n(single_summaries[name], documents[name], 1)[1],
                              "vector f1": vector_score(single_summaries[name], documents[name])[2]})
    print(pd.DataFrame(self_rows).to_string(index=False))

In [ ]:
title("Operation 16: Data drift check and retraining")


def word_distribution(texts, vocabulary):
    counts = Counter()
    for text in texts:
        counts.update(get_tokens(text))
    values = np.array([counts[word] for word in vocabulary], dtype=float) + 1
    return values / values.sum()


def check_drift(old_texts, new_texts, limit=0.35):
    old_words = set(word for text in old_texts for word in get_tokens(text))
    new_words = [word for text in new_texts for word in get_tokens(text)]
    vocabulary = sorted(old_words | set(new_words))
    distance = float(jensenshannon(word_distribution(old_texts, vocabulary), word_distribution(new_texts, vocabulary)))
    unknown_ratio = sum(1 for word in new_words if word not in old_words) / max(len(new_words), 1)
    return {"distance": round(distance, 3), "new word ratio": round(unknown_ratio, 3), "drift": bool(distance > limit or unknown_ratio > 0.5)}


drift_rows = []
for group in version_groups:
    drift_rows.append({"check": f"{group[0]} -> {group[-1]}", **check_drift([documents[group[0]]], [documents[group[-1]]])})
drift_table = pd.DataFrame(drift_rows, columns=["check", "distance", "new word ratio", "drift"])
print("Drift between the oldest and newest version of each document:")
print(drift_table.to_string(index=False) if not drift_table.empty else "    no version groups to check")

unusual_rows = []
for name in names:
    others = [documents[other] for other in names if other != name]
    if others:
        result = check_drift(others, [documents[name]])
        unusual_rows.append({"document": name, "distance from the rest": result["distance"], "new word ratio": result["new word ratio"]})
unusual_table = pd.DataFrame(unusual_rows, columns=["document", "distance from the rest", "new word ratio"])
unusual_table = unusual_table.sort_values("distance from the rest", ascending=False).reset_index(drop=True)
print()
print("How different each document is from all the others:")
print(unusual_table.to_string(index=False))
if not unusual_table.empty:
    print("Most unusual document:", unusual_table.iloc[0]["document"])

print()
if not drift_table.empty and drift_table["drift"].any():
    print()
    print("Drift found, so the vectorizer is trained again on all the text")
    start = time.perf_counter()
    main_vectorizer = TextVectorizer().fit(all_sentences + [documents[name] for name in names])
    print(f"Retraining finished in {time.perf_counter() - start:.2f} seconds, vocabulary size {len(main_vectorizer.word_model.vocabulary_)}")
else:
    print("No drift found, the current model is fine")

In [ ]:
title("Operation 17: Automatic tests and speed check")


def measure_time(function, *arguments, rounds=3):
    times = []
    for _ in range(rounds):
        start = time.perf_counter()
        function(*arguments)
        times.append(time.perf_counter() - start)
    return round(min(times) * 1000, 2), round(sum(times) / len(times) * 1000, 2)


def test_tokens():
    assert get_tokens("The cats are running") == ["cat", "running"]


def test_sentence_split_joins_broken_lines():
    text = "This is a long line that was broken by the PDF reader in the middle of the\nsentence and it continues here."
    assert len(split_sentences(text)) == 1


def test_polish_verb_rejected():
    assert "Polish Language" not in extract_skills("We hand-polish the UI before release.")[0]


def test_frontend_scripting_is_javascript():
    assert "JavaScript" in extract_skills("Comfortable with frontend scripting.")[0]


def test_alignment_same_text():
    lines = ["Intro to the course.", "Tokenization basics.", "Language models and smoothing."]
    steps, score = needleman_wunsch(lines, lines, main_vectorizer)
    assert all(step[0] == "SAME" for step in steps)


def test_alignment_finds_added_line():
    old = ["Intro to the course.", "Language models and smoothing."]
    new = ["Intro to the course.", "Word embeddings such as GloVe.", "Language models and smoothing."]
    steps, score = needleman_wunsch(old, new, main_vectorizer)
    assert [step[0] for step in steps] == ["SAME", "ADDED", "SAME"]


def test_number_change_is_not_same():
    steps, score = needleman_wunsch(["Employees get 12 days of leave."], ["Employees get 15 days of leave."], main_vectorizer)
    assert steps[0][0] == "CHANGED"


def test_spelling_errors_still_same():
    steps, score = needleman_wunsch(["Text preprocessing including tokenization and stopword removal."],
                                    ["Text preprocesing includng tokenizaton and stopword remval."], main_vectorizer)
    assert steps[0][0] == "SAME"


def test_numbering_ignored():
    assert strip_numbering("Unit 4: Text classification.") == strip_numbering("Unit 3: Text classification.")


def test_summary_is_shorter():
    longest = max(names, key=lambda name: len(documents[name]))
    summary = summarize([longest])[0]
    assert len(summary.split()) <= len(documents[longest].split())


def test_rouge_perfect_match():
    assert rouge_n("the cat sat on the mat", "the cat sat on the mat", 1)[2] == 1.0


def test_fairness():
    assert fairness_table.empty or fairness_table["fair"].all()


tests = [test_tokens, test_sentence_split_joins_broken_lines, test_polish_verb_rejected, test_frontend_scripting_is_javascript,
         test_alignment_same_text, test_alignment_finds_added_line, test_number_change_is_not_same,
         test_spelling_errors_still_same, test_numbering_ignored, test_summary_is_shorter,
         test_rouge_perfect_match, test_fairness]
test_rows = []
for test in tests:
    try:
        test()
        test_rows.append({"test": test.__name__, "result": "PASSED"})
    except AssertionError:
        test_rows.append({"test": test.__name__, "result": "FAILED"})
test_table = pd.DataFrame(test_rows)
print(test_table.to_string(index=False))
print()

first_doc = names[0]
benchmark_jobs = [
    ("vector search", store.search, ("main idea of the document",)),
    ("skill extraction", extract_skills, (documents[first_doc],)),
    ("summary", summarize, ([first_doc],)),
]
if version_groups:
    benchmark_jobs.append(("version comparison", compare_documents, (version_groups[0][0], version_groups[0][1], False)))
if job_file and resume_files:
    benchmark_jobs.append(("resume match", match_resume, (documents[resume_files[0]], documents[job_file])))
benchmark_rows = []
for label, function, arguments in benchmark_jobs:
    fastest, average = measure_time(function, *arguments)
    benchmark_rows.append({"task": label, "fastest ms": fastest, "average ms": average})
benchmark_table = pd.DataFrame(benchmark_rows)
print(benchmark_table.to_string(index=False))

In [ ]:
title("Operation 18: PDF report")
styles = getSampleStyleSheet()
title_style = styles["Title"]
heading_style = styles["Heading2"]
small_heading = styles["Heading3"]
body_style = ParagraphStyle("body", parent=styles["BodyText"], fontSize=9, leading=12)
summary_style = ParagraphStyle("summary", parent=styles["BodyText"], fontSize=10.5, leading=21)
added_style = ParagraphStyle("added", parent=body_style, backColor=colors.HexColor("#d9f2d9"), borderPadding=3, spaceAfter=5)
removed_style = ParagraphStyle("removed", parent=body_style, backColor=colors.HexColor("#f8d7d7"), borderPadding=3, spaceAfter=5)
changed_style = ParagraphStyle("changed", parent=body_style, backColor=colors.HexColor("#fff1cc"), borderPadding=3, spaceAfter=5)
same_style = ParagraphStyle("same", parent=body_style, textColor=colors.grey, spaceAfter=5)
PAGE_WIDTH = A4[0] - 80


def safe(text):
    return escape(str(text))


def cell(text):
    return Paragraph(safe(text), body_style)


def make_table(frame, widths=None):
    rows = [[cell(column) for column in frame.columns]]
    for values in frame.values.tolist():
        rows.append([cell(value) for value in values])
    if widths is None:
        widths = [PAGE_WIDTH / len(frame.columns)] * len(frame.columns)
    table = Table(rows, colWidths=widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#d6e2ee")),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f4f4f4")]),
    ]))
    return table


def add_comparison(story, result):
    story.append(Paragraph(safe(f"{result['old']} to {result['new']}"), small_heading))
    story.append(Paragraph(safe(f"Similarity {result['similarity']}. {result['relation']}. Changes: {result['counts']}. "
                                f"A normal line diff would flag {result['lexical lines flagged']} lines."), body_style))
    if result["numbers added"] or result["numbers removed"]:
        story.append(Paragraph(safe(f"Numbers added: {', '.join(result['numbers added']) or 'none'}. Numbers removed: {', '.join(result['numbers removed']) or 'none'}."), body_style))
    story.append(Spacer(1, 4))
    for label, old_sentence, new_sentence, value in result["steps"]:
        if label == "ADDED":
            story.append(Paragraph("<b>ADDED:</b> " + safe(new_sentence), added_style))
        elif label == "REMOVED":
            story.append(Paragraph("<b>REMOVED:</b> " + safe(old_sentence), removed_style))
        elif label == "CHANGED":
            story.append(Paragraph(f"<b>CHANGED ({value:.2f}):</b> {safe(old_sentence)}<br/><b>NOW:</b> {safe(new_sentence)}", changed_style))
        else:
            story.append(Paragraph("SAME: " + safe(new_sentence), same_style))


def build_report(file_name):
    path = os.path.join(OUTPUT_FOLDER, file_name)
    document = SimpleDocTemplate(path, pagesize=A4, leftMargin=40, rightMargin=40, topMargin=40, bottomMargin=40)
    story = [Paragraph("Multi Document NLP Scanner Report", title_style),
             Paragraph(safe(f"Created on {time.strftime('%d %B %Y, %H:%M')} for {len(names)} documents"), body_style), Spacer(1, 10)]

    story.append(Paragraph("1. Document overview", heading_style))
    story.append(make_table(overview_table[["document", "words", "sentences", "readability", "level", "mood", "top keywords"]],
                            [95, 40, 50, 55, 50, 50, PAGE_WIDTH - 340]))
    story.append(Spacer(1, 8))
    story.append(Paragraph("Topics", small_heading))
    story.append(make_table(topic_table, [130, 70, PAGE_WIDTH - 200]))
    story.append(Spacer(1, 8))
    story.append(Paragraph("Similarity between documents", small_heading))
    image_width = PAGE_WIDTH * 0.85
    story.append(PdfImage(heatmap_path, width=image_width, height=image_width * figure_size[1] / figure_size[0]))

    story.append(PageBreak())
    story.append(Paragraph("2. Named entities and skills", heading_style))
    entity_rows = []
    for name in names:
        parts = [f"{label}: {', '.join(item for item, count in counter.most_common(4))}" for label, counter in document_entities[name].items() if counter]
        entity_rows.append({"document": name, "entities": "; ".join(parts) or "none", "skills": ", ".join(document_skills[name]) or "none"})
    story.append(make_table(pd.DataFrame(entity_rows), [95, PAGE_WIDTH - 255, 160]))
    if not shared_table.empty:
        story.append(Spacer(1, 8))
        story.append(Paragraph("Shared or copied sentences", small_heading))
        story.append(make_table(shared_table[["similarity", "document A", "sentence A", "document B"]], [50, 90, PAGE_WIDTH - 230, 90]))

    story.append(Spacer(1, 8))
    story.append(Paragraph("3. Semantic search", heading_style))
    search_rows = [{"question": question, "best answer": results[0]["text"] if results else "", "source": results[0]["source"] if results else "",
                    "score": results[0]["score"] if results else 0} for question, results in search_results.items()]
    story.append(make_table(pd.DataFrame(search_rows), [130, PAGE_WIDTH - 260, 90, 40]))

    if match_results:
        story.append(PageBreak())
        story.append(Paragraph("4. Resume and job matching", heading_style))
        story.append(Paragraph(safe("Job file: " + job_file + ". Skills needed: " + ", ".join(extract_skills(documents[job_file])[0])), body_style))
        story.append(Spacer(1, 4))
        match_rows = [{"rank": rank + 1, "resume": name, "final": match_results[name]["final_score"], "skills": match_results[name]["skill_score"],
                       "coverage": match_results[name]["coverage"], "matched": ", ".join(match_results[name]["matched"]) or "none",
                       "missing": ", ".join(match_results[name]["missing"]) or "none"} for rank, name in enumerate(ranking["resume"])]
        story.append(make_table(pd.DataFrame(match_rows), [30, 70, 35, 35, 50, (PAGE_WIDTH - 220) / 2, (PAGE_WIDTH - 220) / 2]))
        if not fairness_table.empty:
            story.append(Spacer(1, 6))
            story.append(Paragraph("Fairness check (name and gender swap)", small_heading))
            story.append(make_table(fairness_table))

    if version_results:
        story.append(PageBreak())
        story.append(Paragraph("5. Semantic version comparison", heading_style))
        story.append(Paragraph("Green means added, red means removed, yellow means changed and grey means the same.", body_style))
        story.append(Spacer(1, 4))
        story.append(make_table(evolution_table))
        for result in version_results:
            add_comparison(story, result)

    story.append(PageBreak())
    story.append(Paragraph("6. Summaries", heading_style))
    for item in group_summaries:
        story.append(Paragraph(safe("Documents: " + ", ".join(item["files"])), small_heading))
        story.append(Paragraph(safe(item["summary"]), summary_style))
        story.append(make_table(pd.DataFrame(item["trace"], columns=["evidence sentence", "source"]), [PAGE_WIDTH - 110, 110]))
        story.append(Spacer(1, 6))
    if not evaluation.empty:
        story.append(Paragraph(safe("Quality against " + reference_file), small_heading))
        story.append(make_table(evaluation.reset_index().rename(columns={"index": "metric"})))
    story.append(Spacer(1, 6))
    story.append(Paragraph("Single document summaries", small_heading))
    story.append(make_table(pd.DataFrame(list(single_summaries.items()), columns=["document", "summary"]), [110, PAGE_WIDTH - 110]))

    story.append(PageBreak())
    story.append(Paragraph("7. Drift, tests and speed", heading_style))
    if not drift_table.empty:
        story.append(make_table(drift_table, [PAGE_WIDTH - 210, 70, 80, 60]))
        story.append(Spacer(1, 6))
    if not unusual_table.empty:
        story.append(make_table(unusual_table))
    story.append(Spacer(1, 6))
    story.append(make_table(test_table))
    story.append(Spacer(1, 6))
    story.append(make_table(benchmark_table))

    document.build(story)
    return path


report_path = build_report("nlp_scanner_report.pdf")
print("Report saved to", os.path.abspath(report_path))

In [ ]:
import math
import networkx as nx
from nltk.corpus import wordnet
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
from sklearn.decomposition import PCA


def rake_keyphrases(text, top=10, longest=4):
    phrases = []
    for sentence in split_sentences(text):
        pieces = re.findall(r"[a-z][a-z+#]*|[,;:()\[\]!?]", sentence.lower())
        current = []
        for piece in pieces:
            if not piece[0].isalpha() or piece in stop_words:
                if current:
                    phrases.append(current)
                current = []
            else:
                current.append(piece)
        if current:
            phrases.append(current)
    phrases = [phrase for phrase in phrases if len(phrase) <= longest]
    frequency = Counter()
    degree = Counter()
    for phrase in phrases:
        for word in phrase:
            frequency[word] += 1
            degree[word] += len(phrase)
    phrase_scores = {}
    for phrase in phrases:
        key = " ".join(phrase)
        phrase_scores[key] = max(phrase_scores.get(key, 0), sum(degree[word] / frequency[word] for word in phrase))
    best = sorted(phrase_scores.items(), key=lambda item: item[1], reverse=True)[:top]
    return [(phrase, round(score, 2)) for phrase, score in best]


def word_pairs(text, top=10):
    words = [word for word in word_tokenize(text.lower()) if word.isalpha()]
    finder = BigramCollocationFinder.from_words(words)
    finder.apply_word_filter(lambda word: word in stop_words or len(word) < 3)
    if len(words) > 400:
        finder.apply_freq_filter(2)
    scored = finder.score_ngrams(BigramAssocMeasures().likelihood_ratio)[:top]
    return [(" ".join(pair), round(score, 2), finder.ngram_fd[pair]) for pair, score in scored]


title("Operation 19: RAKE key phrases and strong word pairs")
for name in names[:4]:
    print(name)
    print("    Key phrases:", ", ".join(f"{phrase} ({score})" for phrase, score in rake_keyphrases(documents[name], top=5)))
    print("    Word pairs :", ", ".join(pair for pair, score, count in word_pairs(documents[name], top=5)))

In [ ]:
document_type_clues = {
    "Resume or CV": "resume curriculum vitae experience skills education project worked built developed email phone linkedin summary internship",
    "Job description": "job role responsibilities requirements candidate looking hiring apply qualifications preferred required salary position join",
    "Policy or legal": "policy shall must employee employees rules compliance agreement terms conditions clause leave applies company law allowed",
    "Academic or syllabus": "course unit syllabus students semester exam assessment lecture module credits learning outcomes chapter introduction",
    "News or article": "said reported according year growing world government people million percent new country market expected",
    "Research paper": "abstract method methods results experiment dataset proposed model evaluation conclusion related work accuracy paper study",
    "Technical document": "system software architecture api database server module function requirement design implementation code pipeline",
    "Business or finance": "revenue profit sales cost budget quarter financial market growth investment customers price",
    "Letter or email": "dear regards sincerely thank please request meeting attached hello kindly",
}


def detect_document_type(text):
    words = Counter(stemmer.stem(word) for word in get_tokens(text))
    total = sum(words.values()) or 1
    scores = {}
    for label, clue_text in document_type_clues.items():
        clues = set(stemmer.stem(word) for word in clue_text.split())
        scores[label] = sum(words[clue] for clue in clues) / total
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    best_label, best_score = ranked[0]
    confidence = best_score / (sum(score for label, score in ranked) or 1)
    return best_label, round(confidence, 2), [(label, round(score, 3)) for label, score in ranked]


def detect_language(text):
    words = set(word for word in word_tokenize(text.lower()) if word.isalpha())
    scores = {}
    for language in stopwords.fileids():
        if language == "hinglish":
            continue
        base = language.split("-")[0]
        scores[base] = max(scores.get(base, 0), len(words & set(stopwords.words(language))))
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    total = sum(score for language, score in ranked[:3]) or 1
    return ranked[0][0].title(), round(ranked[0][1] / total, 2), [(language.title(), score) for language, score in ranked[:5]]


title("Operation 20: Document type and language detection")
type_rows = []
for name in names:
    doc_type, type_confidence, type_scores = detect_document_type(documents[name])
    language, language_confidence, language_scores = detect_language(documents[name])
    type_rows.append({"document": name, "detected type": doc_type, "type confidence": type_confidence,
                      "language": language, "language confidence": language_confidence})
print(pd.DataFrame(type_rows).to_string(index=False))
print()
print("Language test on a French sentence:", detect_language("Le chat est sur la table et il dort avec le chien.")[0])

In [ ]:
pos_groups = {"Nouns": "NN", "Verbs": "VB", "Adjectives": "JJ", "Adverbs": "RB", "Pronouns": "PRP", "Numbers": "CD"}


def style_profile(text):
    tokens = [word for word in word_tokenize(text) if any(letter.isalnum() for letter in word)]
    counts = Counter()
    for word, tag in nltk.pos_tag(tokens):
        group = next((name for name, start in pos_groups.items() if tag.startswith(start)), "Others")
        counts[group] += 1
    words = [word for word in tokens if word.isalpha()]
    unique = set(word.lower() for word in words)
    return {
        "pos counts": dict(counts),
        "lexical diversity": round(len(unique) / max(len(words), 1), 3),
        "average word length": round(sum(len(word) for word in words) / max(len(words), 1), 2),
        "long word ratio": round(sum(1 for word in words if len(word) >= 7) / max(len(words), 1), 3),
    }


def readability_scores(text):
    sentences = split_sentences(text)
    words = [word for word in word_tokenize(text) if word.isalpha()]
    if not sentences or not words:
        return {"Flesch reading ease": 0.0, "Flesch-Kincaid grade": 0.0, "Gunning fog": 0.0, "SMOG index": 0.0}
    syllables = [count_syllables(word) for word in words]
    words_per_sentence = len(words) / len(sentences)
    syllables_per_word = sum(syllables) / len(words)
    complex_words = sum(1 for count in syllables if count >= 3)
    return {
        "Flesch reading ease": round(206.835 - 1.015 * words_per_sentence - 84.6 * syllables_per_word, 1),
        "Flesch-Kincaid grade": round(0.39 * words_per_sentence + 11.8 * syllables_per_word - 15.59, 1),
        "Gunning fog": round(0.4 * (words_per_sentence + 100 * complex_words / len(words)), 1),
        "SMOG index": round(1.043 * math.sqrt(complex_words * 30 / len(sentences)) + 3.1291, 1),
    }


title("Operation 21: Writing style and four readability formulas")
style_rows = []
for name in names:
    profile = style_profile(documents[name])
    style_rows.append({"document": name, **readability_scores(documents[name]),
                       "lexical diversity": profile["lexical diversity"], "avg word length": profile["average word length"],
                       "nouns": profile["pos counts"].get("Nouns", 0), "verbs": profile["pos counts"].get("Verbs", 0),
                       "adjectives": profile["pos counts"].get("Adjectives", 0)})
print(pd.DataFrame(style_rows).to_string(index=False))

In [ ]:
def mood_of(score):
    return "Positive" if score > 0.05 else "Negative" if score < -0.05 else "Neutral"


def sentiment_arc(text):
    rows = []
    for index, sentence in enumerate(split_sentences(text), 1):
        score = sentiment_tool.polarity_scores(sentence)["compound"]
        rows.append({"position": index, "sentence": sentence, "score": score, "mood": mood_of(score)})
    return pd.DataFrame(rows, columns=["position", "sentence", "score", "mood"])


title("Operation 22: Sentiment arc, sentence by sentence")
arc_document = max(names, key=lambda name: len(document_sentences[name]))
arc = sentiment_arc(documents[arc_document])
print("Document:", arc_document)
print(arc.to_string(index=False, max_colwidth=70))

In [ ]:
question_types = [("how many", "NUMBER"), ("how much", "NUMBER"), ("what percent", "NUMBER"), ("when", "DATE"),
                  ("what year", "DATE"), ("who", "PERSON"), ("where", "PLACE"), ("which company", "ORGANIZATION")]
number_answer = r"\b\d+(?:[.,]\d+)?\s?(?:%|percent|days?|years?|months?|hours?|weeks?|people|million|billion)?"


def question_type(question):
    lowered = question.lower()
    for start, kind in question_types:
        if start in lowered:
            return kind
    return "GENERAL"


def answer_question(question, store, top_k=3, only_from=None):
    results = store.search(question, top_k=top_k, only_from=only_from)
    if not results:
        return {"question type": question_type(question), "short answer": "", "answer sentence": "", "source": "",
                "score": 0.0, "matched words": [], "other results": []}
    best = results[0]
    kind = question_type(question)
    short_answer = ""
    if kind == "NUMBER":
        found = [item.strip() for item in re.findall(number_answer, best["text"]) if item.strip()]
        short_answer = found[0] if found else ""
    elif kind == "DATE":
        found = re.findall(month_pattern, best["text"], flags=re.IGNORECASE)
        short_answer = found[0] if found else ""
    elif kind in ("PERSON", "PLACE", "ORGANIZATION"):
        found = find_entities(best["text"])[kind]
        short_answer = found.most_common(1)[0][0] if found else ""
    matched = sorted(set(get_tokens(question)) & set(get_tokens(best["text"])))
    return {"question type": kind, "short answer": short_answer, "answer sentence": best["text"], "source": best["source"],
            "score": best["score"], "matched words": matched, "other results": results[1:]}


title("Operation 23: Question answering with short answers")
for question in SEARCH_QUESTIONS:
    answer = answer_question(question, store)
    print("Question     :", question)
    print("Type         :", answer["question type"])
    print("Short answer :", answer["short answer"] or "(see sentence)")
    print("From         :", f"[{answer['source']}] {answer['answer sentence']}")
    print()

In [ ]:
def redact_text(text):
    counts = Counter()

    def mask(pattern, label, value, flags=0):
        counts[label] += len(re.findall(pattern, value, flags))
        return re.sub(pattern, f"[{label}]", value, flags=flags)

    text = mask(r"[\w.+-]+@[\w-]+\.[\w.]+", "EMAIL", text)
    text = mask(r"https?://\S+|www\.\S+", "URL", text)
    text = mask(r"\+?\d[\d\s-]{8,}\d", "PHONE", text)
    name_lines = re.findall(r"(?im)^\s*name\s*:\s*(.+)$", text)
    people = set(name.strip() for name in name_lines) | set(find_entities(text)["PERSON"].keys())
    for person in sorted(people, key=len, reverse=True):
        if person and person in text:
            counts["PERSON"] += text.count(person)
            text = text.replace(person, "[PERSON]")
    return text, {label: count for label, count in counts.items() if count}


def shingles(text, size=4):
    words = [word for word in word_tokenize(text.lower()) if word.isalnum()]
    return set(tuple(words[i:i + size]) for i in range(len(words) - size + 1))


def copy_check(text_a, text_b, size=4):
    first, second = shingles(text_a, size), shingles(text_b, size)
    if not first or not second:
        return {"jaccard": 0.0, "A found in B": 0.0, "B found in A": 0.0, "shared phrases": []}
    common = first & second
    return {"jaccard": round(len(common) / len(first | second), 3), "A found in B": round(len(common) / len(first), 3),
            "B found in A": round(len(common) / len(second), 3), "shared phrases": sorted(" ".join(item) for item in common)[:15]}


title("Operation 24: Privacy redaction and copy check")
resume_example = next((name for name in names if "resume" in name.lower()), names[0])
redacted, redaction_counts = redact_text(documents[resume_example])
print("Redacted version of", resume_example)
print(redacted)
print("Hidden items:", redaction_counts)
print()
if len(names) >= 2:
    first_name, second_name = (version_groups[0][0], version_groups[0][1]) if version_groups else (names[0], names[1])
    print(f"Copy check between {first_name} and {second_name}:")
    print(copy_check(documents[first_name], documents[second_name]))

In [ ]:
triple_grammar = nltk.RegexpParser(r"""
    NP: {<DT|PRP\$|CD>?<JJ.*|VBN|VBG>*<NN.*|PRP>+}
    VP: {<MD>?<RB.*>?<VB.*>+<RB.*|RP|IN|TO>?}
""")


pronouns = {"i", "we", "you", "he", "she", "it", "they", "me", "us", "them", "this", "that"}


def find_triples(text, max_sentences=300):
    triples = []
    for sentence in split_sentences(text)[:max_sentences]:
        tree = triple_grammar.parse(nltk.pos_tag(word_tokenize(sentence)))
        chunks = []
        for part in tree:
            if hasattr(part, "label"):
                chunks.append((part.label(), " ".join(word for word, tag in part.leaves())))
            else:
                chunks.append(("O", part[0]))
        for i in range(len(chunks) - 2):
            if chunks[i][0] == "NP" and chunks[i + 1][0] == "VP" and chunks[i + 2][0] == "NP":
                if chunks[i][1].lower() in pronouns or chunks[i + 2][1].lower() in pronouns:
                    continue
                if not re.search(r"[a-z]", chunks[i + 1][1].lower()) or "@" in chunks[i][1] + chunks[i + 2][1]:
                    continue
                triples.append({"subject": chunks[i][1], "relation": chunks[i + 1][1], "object": chunks[i + 2][1], "sentence": sentence})
    return pd.DataFrame(triples, columns=["subject", "relation", "object", "sentence"])


def clean_node(phrase):
    words = [lemmatizer.lemmatize(word) for word in phrase.lower().split() if word not in {"a", "an", "the", "this", "that", "these", "those", "their", "its", "our", "your"}]
    return " ".join(words) or phrase.lower()


def build_graph(triples):
    graph = nx.DiGraph()
    for row in triples.itertuples():
        graph.add_edge(clean_node(row.subject), clean_node(row.object), label=row.relation.lower())
    return graph


title("Operation 25: Knowledge graph from subject, verb and object triples")
all_triples = pd.concat([find_triples(documents[name]).assign(document=name) for name in names], ignore_index=True)
print(all_triples[["document", "subject", "relation", "object"]].head(15).to_string(index=False))
knowledge_graph = build_graph(all_triples)
print()
print("Graph nodes:", knowledge_graph.number_of_nodes(), "| Graph links:", knowledge_graph.number_of_edges())
if knowledge_graph.number_of_nodes():
    central = sorted(knowledge_graph.degree, key=lambda item: item[1], reverse=True)[:5]
    print("Most connected ideas:", ", ".join(f"{node} ({degree})" for node, degree in central))

In [ ]:
wordnet_types = {"n": "noun", "v": "verb", "a": "adjective", "s": "adjective", "r": "adverb"}


def word_meaning(word, limit=4):
    rows = []
    for synset in wordnet.synsets(word.strip().replace(" ", "_"))[:limit]:
        lemmas = synset.lemmas()
        rows.append({
            "type": wordnet_types.get(synset.pos(), synset.pos()),
            "meaning": synset.definition(),
            "synonyms": ", ".join(sorted({lemma.name().replace("_", " ") for lemma in lemmas})[:6]),
            "opposites": ", ".join(sorted({opposite.name().replace("_", " ") for lemma in lemmas for opposite in lemma.antonyms()})),
            "broader idea": ", ".join(hyper.lemmas()[0].name().replace("_", " ") for hyper in synset.hypernyms()[:2]),
            "example": synset.examples()[0] if synset.examples() else "",
        })
    return pd.DataFrame(rows, columns=["type", "meaning", "synonyms", "opposites", "broader idea", "example"])


title("Operation 26: Word meanings with WordNet")
for word in [keyword for keyword in top_keywords(names[0], 6) if " " not in keyword][:2]:
    print("Word:", word)
    print(word_meaning(word, limit=3).to_string(index=False, max_colwidth=50))
    print()

In [ ]:
tag_meanings = {
    "NN": "noun", "NNS": "plural noun", "NNP": "proper noun", "NNPS": "plural proper noun", "VB": "verb", "VBD": "past tense verb",
    "VBG": "verb ending in ing", "VBN": "past participle", "VBP": "present verb", "VBZ": "present verb", "JJ": "adjective",
    "JJR": "comparing adjective", "JJS": "most adjective", "RB": "adverb", "RBR": "comparing adverb", "PRP": "pronoun",
    "PRP$": "possessive pronoun", "DT": "determiner", "IN": "preposition", "CC": "joining word", "CD": "number", "MD": "modal verb",
    "TO": "to", "UH": "interjection", "WP": "question word", "WDT": "question word", "WRB": "question word", "RP": "particle",
    "EX": "there", "POS": "possessive ending",
}


def wordnet_tag(tag):
    if tag.startswith("J"):
        return "a"
    if tag.startswith("V"):
        return "v"
    if tag.startswith("R"):
        return "r"
    return "n"


def nlp_pipeline(sentence):
    tagged = nltk.pos_tag(word_tokenize(sentence))
    tree = entity_chunker.parse(tagged)
    entity_labels = []
    for part in tree:
        if hasattr(part, "label"):
            entity_labels.extend([part.label()] * len(part.leaves()))
        else:
            entity_labels.append("")
    rows = []
    for (word, tag), entity in zip(tagged, entity_labels):
        rows.append({"token": word, "lowercase": word.lower(), "stopword": "yes" if word.lower() in stop_words else "no",
                     "stem": stemmer.stem(word.lower()), "lemma": lemmatizer.lemmatize(word.lower(), wordnet_tag(tag)),
                     "POS tag": tag, "meaning": tag_meanings.get(tag, "punctuation or symbol"), "entity": entity})
    return pd.DataFrame(rows)


title("Operation 27: Step by step NLP pipeline for one sentence")
print(nlp_pipeline("Sara Khan was building better dashboards for Google in London last year.").to_string(index=False))

In [ ]:
def sentence_map(vectorizer, sentence_items):
    if len(sentence_items) < 3:
        return pd.DataFrame(columns=["x", "y", "document", "sentence"])
    vectors = vectorizer.transform([sentence for source, sentence in sentence_items])
    points = PCA(n_components=2, random_state=0).fit_transform(vectors)
    return pd.DataFrame({"x": points[:, 0], "y": points[:, 1], "document": [source for source, sentence in sentence_items],
                         "sentence": [sentence for source, sentence in sentence_items]})


def word_frequency(text, top=20):
    return Counter(get_tokens(text)).most_common(top)


def zipf_table(text):
    counts = Counter(get_tokens(text)).most_common()
    return pd.DataFrame({"rank": range(1, len(counts) + 1), "word": [word for word, count in counts], "count": [count for word, count in counts]})


title("Operation 28: Sentence map and word frequency")
map_items = [(name, sentence) for name in names for sentence in document_sentences[name]]
points = sentence_map(main_vectorizer, map_items)
figure, axis = plt.subplots(figsize=(9, 6))
for name, group in points.groupby("document"):
    axis.scatter(group["x"], group["y"], label=name, s=40, alpha=0.75)
axis.set_title("Every sentence placed by meaning (PCA of TF-IDF vectors)")
axis.legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()
all_text = " ".join(documents.values())
print("Most common words in all documents:", ", ".join(f"{word} ({count})" for word, count in word_frequency(all_text, 12)))